# World Cup 2026 - Task 2.2: regression
Linear regression on the processed dataset (208 team-match rows, 8 pre-match variables).
Input: `wc2026_2_2_dataset.csv`, produced by the dataset processing notebook.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# the processed dataset
DATA_FILE = "wc2026_2_2_dataset.csv"

## 2. Load and check

In [ ]:
df = pd.read_csv(DATA_FILE)

# the 8 explanatory variables and the target
X_COLS = ["fifa_points_team", "squad_value_team_eur_m", "squad_value_opp_eur_m",
          "form_gf_pg_team", "form_ga_pg_opp", "rest_days", "is_host", "is_knockout"]
Y_COL = "goals_scored"   # goals scored by the team in the match

assert len(df) == 208                                   # 208 team-match rows
assert len(X_COLS) == 8                                 # 8 explanatory variables
assert df[X_COLS + [Y_COL]].isna().sum().sum() == 0     # nothing missing
print("All checks passed")
df[X_COLS + [Y_COL]].describe().round(2)

## 3. Linear regression
Variables with a p-value under 0.05 are the ones that clearly help the prediction.

In [ ]:
X = df[X_COLS]
y = df[Y_COL]
model = sm.OLS(y, sm.add_constant(X)).fit()
print(model.summary())

## 4. Multicollinearity check
VIF (variance inflation factor) shows whether two variables carry the same information.
Values under 5 are fine. FIFA points and squad value are the most related pair, as expected.

In [ ]:
Xc = sm.add_constant(X)
vif = pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
                index=X_COLS, name="VIF")
vif.round(2).sort_values(ascending=False)

## 5. Cross-validation
Trains on 80% of the matches and predicts the other 20%, five times, to see how it does on unseen matches.
Both rows of a match always go into the same fold (`GroupKFold` on `match_id`), so the model
never sees one side of a match while being tested on the other.
The baseline always predicts the average number of goals, so the model has to beat it to be useful.

In [ ]:
cv = GroupKFold(n_splits=5)
pred = cross_val_predict(LinearRegression(), X, y, cv=cv, groups=df["match_id"])
baseline = np.full(len(y), y.mean())

print(f"CV R2  : {r2_score(y, pred):.3f}")
print(f"CV MAE : {mean_absolute_error(y, pred):.3f} goals   (baseline {mean_absolute_error(y, baseline):.3f})")
print(f"CV RMSE: {np.sqrt(mean_squared_error(y, pred)):.3f} goals   (baseline {np.sqrt(mean_squared_error(y, baseline)):.3f})")

## 6. Visualizations

In [ ]:
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e5e5", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "figure.dpi": 110,
})
BLUE, ORANGE, GRAY = "#2f6db5", "#d9622b", "#6b7280"

### Chart 1: squad value vs goals scored

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

# one dot per team per match (a little vertical jitter so stacked dots can be seen)
rng = np.random.default_rng(0)
ax.scatter(df["squad_value_team_eur_m"], df[Y_COL] + rng.uniform(-0.12, 0.12, len(df)),
           s=30, color=BLUE, alpha=0.6, edgecolor="white", linewidth=0.6)

# straight line that best fits the dots
slope, intercept = np.polyfit(df["squad_value_team_eur_m"], df[Y_COL], 1)
xs = np.linspace(df["squad_value_team_eur_m"].min(), df["squad_value_team_eur_m"].max(), 100)
ax.plot(xs, slope * xs + intercept, color=ORANGE, linewidth=2)

r = np.corrcoef(df["squad_value_team_eur_m"], df[Y_COL])[0, 1]
ax.text(0.03, 0.95, f"correlation r = {r:.2f}", transform=ax.transAxes, va="top", color=GRAY)
ax.set_title("More valuable squads tend to score more goals", loc="left")
ax.set_xlabel("Team squad value (EUR millions)")
ax.set_ylabel("Goals scored in the match")
plt.savefig("squad value vs goals scored.png", bbox_inches="tight")
plt.show()

### Chart 2: actual vs predicted goals

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))

# each row's prediction comes from the cross-validation step
ax.scatter(y + rng.uniform(-0.12, 0.12, len(y)), pred, s=30, color=BLUE, alpha=0.6,
           edgecolor="white", linewidth=0.6)

# on this line, prediction = actual
lim = [min(y.min(), pred.min()) - 0.5, max(y.max(), pred.max()) + 0.5]
ax.plot(lim, lim, color=ORANGE, linewidth=2)
ax.set_xlim(lim); ax.set_ylim(lim)

ax.set_title("Model predictions vs what actually happened", loc="left")
ax.set_xlabel("Actual goals scored")
ax.set_ylabel("Predicted goals (cross-validated)")
plt.savefig("actual vs predicted goals.png", bbox_inches="tight")
plt.show()

### Chart 3: how much each variable matters

In [ ]:
# variables rescaled to the same units so the bars can be compared
Xz = (X - X.mean()) / X.std()
mz = sm.OLS(y, sm.add_constant(Xz)).fit()
coef = mz.params.drop("const")
ci = mz.conf_int().drop("const")
order = coef.abs().sort_values().index   # smallest effect at the bottom

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(order, coef[order], color=BLUE, height=0.55)

# line on each bar = 95% confidence interval
ax.errorbar(coef[order], range(len(order)),
            xerr=[coef[order] - ci.loc[order, 0], ci.loc[order, 1] - coef[order]],
            fmt="none", ecolor=GRAY, capsize=3, linewidth=1.2)

ax.axvline(0, color=GRAY, linewidth=0.8)
ax.set_title("Effect of each variable on goals scored", loc="left")
ax.set_xlabel("Change in goals scored per 1 std. dev. of the variable")
plt.savefig("how much each variable matters.png", bbox_inches="tight")
plt.show()